# CLOVA 예산 확보 노트북 — 30만원에 과제를 맞춘다

크레딧은 **총 30만원**(과제 20만 + 웰컴 10만), 유효기간 **2026-09-30**, **초과분은 자체 부담**이다.
방침은 [../docs/clova_payment.md](../docs/clova_payment.md), 코퍼스 규모 근거는
[../paper/idea/idea.md](../paper/idea/idea.md) §2.

## 이 노트북이 하는 일

```
총비용 =  벌크 인덱싱 토큰 × 단가          ← 오프라인 (가장 비쌈)
        + 개발셋 질의 × 반복 × 질문당 호출 × 단가
        + 평가 질의 수 × 질문당 호출 × 단가
        + 서버 가동일 × 일단가
        ≤ 300,000원
```

미지수 4개를 **실측해서 CONFIG에 넣으면** 어디를 조여야 하는지 숫자로 나온다.

> ⚠️ **CONFIG의 단가는 전부 가정값이다.** 실측 전 결과는 상대 비교용으로만 읽어라.
> `MEASURED` 플래그가 모두 `True`가 되기 전에는 절대 금액을 신뢰하지 말 것.

**실행 준비**: `pip install ipykernel requests` (pandas 불필요 — 표준 라이브러리만 사용)

In [ ]:
# ============================================================
# CONFIG — 실측한 값으로 바꾸고 MEASURED를 True로 올려라
# ============================================================
BUDGET_KRW = 300_000            # 과제 20만 + 웰컴 10만
CREDIT_EXPIRY = '2026-09-30'

# ---- 미지수 4개 ----
PRICE_IN_PER_1K    = 5.0        # 원 / 입력 1K 토큰   ← NCP 요금표. 아직 가정값
PRICE_OUT_PER_1K   = 10.0       # 원 / 출력 1K 토큰   ← 동일. 아직 가정값
SERVER_KRW_PER_DAY = 2_342.0    # 실측: 시간요금제 92원/h × 24 + 공인IP 4,032원/월 ÷ 30
CHARS_PER_TOKEN_KO = 1.722      # 실측: clova_verify.ipynb §3 (실제 공시 원문 기준)

MEASURED = {
    'PRICE_IN_PER_1K':    False,
    'PRICE_OUT_PER_1K':   False,
    'SERVER_KRW_PER_DAY': True,   # NCP 콘솔 실측
    'CHARS_PER_TOKEN_KO': True,   # clova_verify.ipynb 실측
}

# 서버 요금제 (NCP 콘솔 실측) — 손익분기 66,240 ÷ 92 = 720h = 30일
SERVER_KRW_PER_HOUR  = 92.0     # c2-g3a 2vCPU/4GB, OS 제외
SERVER_KRW_PER_MONTH = 66_240.0
PUBLIC_IP_KRW_MONTH  = 4_032.0

# ---- CLOVA Studio 호출 설정 (§1에서 실측용) ----
CLOVA_API_KEY  = ''            # 발급 직후 1회만 평문 노출된다. 여기 붙이지 말고 환경변수 권장
CLOVA_ENDPOINT = ''            # CLOVA Studio > 플레이그라운드 > '코드 보기'에서 그대로 복사
MODEL = 'HCX-005'

TIERS = ('exchange', 'major', 'holding', 'periodic')

def unmeasured():
    return [k for k, v in MEASURED.items() if not v]

_u = unmeasured()
print('BUDGET:', f'{BUDGET_KRW:,} KRW   (만료 {CREDIT_EXPIRY})')
print('미실측 파라미터:', ', '.join(_u) if _u else '없음 — 절대 금액 신뢰 가능')

---
## 1. 단가 실측

CLOVA Studio 플레이그라운드의 **'코드 보기'** 에서 엔드포인트/헤더를 그대로 복사해
`CLOVA_ENDPOINT`에 넣고 아래를 실행한다. 목적은 두 가지다.

1. **응답의 `usage`** 로 입출력 토큰이 실제로 몇 개 잡히는지 확인
2. 같은 텍스트의 **문자 수 ÷ 토큰 수** 로 `CHARS_PER_TOKEN_KO` 확정

키가 없으면 이 셀은 건너뛰고 문자 기반 추정으로 진행한다.

In [ ]:
import json, os, urllib.request

PROBE_TEXT = (
    '당사는 2025년 3월 14일 이사회 결의를 통해 신규 시설투자를 결정하였습니다. '
    '투자금액은 1,250억원이며 투자기간은 2025년 4월부터 2027년 12월까지입니다. '
    '이는 최근 사업연도 매출액의 8.7%에 해당합니다.'
)

def probe_usage(text=PROBE_TEXT):
    """실제 호출 1회로 usage를 받아 문자/토큰 비율을 잰다."""
    key = CLOVA_API_KEY or os.environ.get('CLOVA_API_KEY', '')
    if not (key and CLOVA_ENDPOINT):
        print('SKIP — CLOVA_API_KEY / CLOVA_ENDPOINT 미설정. 문자 기반 추정으로 진행한다.')
        print(f'  참고: 프로브 텍스트 {len(text)}자 -> 가정 토큰 {len(text)/CHARS_PER_TOKEN_KO:.0f}개')
        return None
    body = json.dumps({
        'messages': [{'role': 'user', 'content': text}],
        'maxTokens': 16, 'temperature': 0.0,
    }).encode('utf-8')
    req = urllib.request.Request(
        CLOVA_ENDPOINT, data=body, method='POST',
        headers={'Content-Type': 'application/json',
                 'Authorization': f'Bearer {key}'},
    )
    with urllib.request.urlopen(req, timeout=60) as r:
        resp = json.loads(r.read().decode('utf-8'))
    print(json.dumps(resp, ensure_ascii=False, indent=2)[:1200])
    # usage 위치는 응답 스펙에 따라 다르다. 눈으로 확인하고 아래를 맞춰라.
    usage = resp.get('usage') or resp.get('result', {}).get('usage') or {}
    pin = usage.get('promptTokens') or usage.get('inputTokens')
    if pin:
        print(f'\n>>> 입력 {len(text)}자 = {pin}토큰  ->  CHARS_PER_TOKEN_KO = {len(text)/pin:.3f}')
        print('    이 값을 CONFIG에 넣고 MEASURED를 True로 올려라.')
    return usage

_ = probe_usage()

---
## 2. 코퍼스 토큰 실측

`data/dart-disclosures/raw`를 실제로 스캔한다. XML은 태그가 대부분이므로
유형별로 표본을 뜯어 **텍스트 비율**을 재고, 전체 바이트에 곱해 추정한다.

In [ ]:
import random, re
from pathlib import Path

TAG = re.compile(r'<[^>]*>')

def find_corpus():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        c = base / 'data' / 'dart-disclosures' / 'raw'
        if c.is_dir():
            return c
    raise FileNotFoundError('data/dart-disclosures/raw 를 찾지 못했다')

def scan_corpus(sample_n=12, seed=42):
    raw, rng, out = find_corpus(), random.Random(seed), {}
    for tier in TIERS:
        d = raw / tier
        if not d.is_dir():
            continue
        files = list(d.rglob('*.xml'))
        if not files:
            continue
        raw_bytes = sum(f.stat().st_size for f in files)
        rs = ts = 0
        for f in rng.sample(files, min(sample_n, len(files))):
            t = f.read_text(encoding='utf-8', errors='ignore')
            rs += len(t); ts += len(TAG.sub(' ', t))
        ratio = ts / rs if rs else 0.0
        chars = raw_bytes * ratio
        out[tier] = {'files': len(files), 'raw_mb': raw_bytes / 1e6, 'ratio': ratio,
                     'chars': chars, 'tokens': chars / CHARS_PER_TOKEN_KO}
    return out

CORPUS = scan_corpus()

print(f"{'tier':<10}{'files':>7}{'raw MB':>9}{'ratio':>7}{'text Mchar':>12}{'M tokens':>10}{'share':>8}")
print('-' * 63)
_tot = sum(v['tokens'] for v in CORPUS.values())
for t, v in CORPUS.items():
    print(f"{t:<10}{v['files']:>7}{v['raw_mb']:>9.0f}{v['ratio']:>7.3f}"
          f"{v['chars']/1e6:>12.0f}{v['tokens']/1e6:>10.0f}{100*v['tokens']/_tot:>7.1f}%")
print('-' * 63)
print(f"{'TOTAL':<10}{'':>7}{'':>9}{'':>7}{'':>12}{_tot/1e6:>10.0f}")
print(f"\n전량 인덱싱 시 입력토큰만 {_tot:,.0f}개 -> {_tot/1000*PRICE_IN_PER_1K:,.0f}원 (출력 별도)")

---
## 3. 비용 모델

`Plan`은 하나의 전략이다. **`extract`** 가 핵심 손잡이 — 유형별로 텍스트의 몇 %를
LLM으로 추출할지 정한다(`0.0` = LLM 안 씀, 규칙 파싱/lazy 로딩으로 처리).

In [ ]:
from dataclasses import dataclass, field

def krw(tok_in, tok_out):
    return tok_in / 1000 * PRICE_IN_PER_1K + tok_out / 1000 * PRICE_OUT_PER_1K

@dataclass
class Plan:
    name: str
    extract: dict                      # tier -> 0.0~1.0 (LLM 추출 비율)
    out_ratio: float = 0.35            # 추출 출력토큰 / 입력토큰
    calls_per_q: int = 6               # 질문당 HCX 호출 수 (파이프라인 상한)
    tok_in_per_call: int = 4_000
    tok_out_per_call: int = 500
    dev_questions: int = 30
    dev_iterations: int = 40           # 개발셋 전체 재실행 횟수
    eval_questions: int = 120          # 비공개 -> 가정
    server_days: int = 20

def cost(p, corpus=None):
    corpus = corpus or CORPUS
    idx_in = sum(corpus[t]['tokens'] * p.extract.get(t, 0.0) for t in corpus)
    per_q  = krw(p.calls_per_q * p.tok_in_per_call, p.calls_per_q * p.tok_out_per_call)
    return {
        'index':  krw(idx_in, idx_in * p.out_ratio),
        'dev':    per_q * p.dev_questions * p.dev_iterations,
        'eval':   per_q * p.eval_questions,
        'server': p.server_days * SERVER_KRW_PER_DAY,
        'per_q':  per_q,
        'idx_tokens': idx_in,
    } | {'total': 0.0}

def total(p, corpus=None):
    c = cost(p, corpus)
    c['total'] = c['index'] + c['dev'] + c['eval'] + c['server']
    return c

def report(plans, corpus=None):
    print(f"{'plan':<26}{'index':>11}{'dev':>10}{'eval':>9}{'server':>9}{'TOTAL':>11}{'':>3}{'여유':>10}")
    print('-' * 89)
    for p in plans:
        c = total(p, corpus)
        ok = 'OK ' if c['total'] <= BUDGET_KRW else 'OVER'
        print(f"{p.name:<26}{c['index']:>11,.0f}{c['dev']:>10,.0f}{c['eval']:>9,.0f}"
              f"{c['server']:>9,.0f}{c['total']:>11,.0f}  {ok}{BUDGET_KRW-c['total']:>10,.0f}")
    print('-' * 89)
    print(f"예산 {BUDGET_KRW:,}원" + ('   ⚠️ 단가 미실측 — 상대 비교용' if unmeasured() else ''))

---
## 4. 시나리오 비교

`idea.md` §5의 3단 차별화 인덱싱을 그대로 시나리오로 옮겼다.
`periodic`은 전체 토큰의 95%를 차지하므로 **여기를 켜는 순간 예산이 무의미해진다.**

In [ ]:
ALL   = {'exchange': 1, 'major': 1, 'holding': 1, 'periodic': 1}
NO_PD = {'exchange': 1, 'major': 1, 'holding': 1, 'periodic': 0}
T1_T2 = {'exchange': 1, 'major': 1, 'holding': 0.3, 'periodic': 0}
T1    = {'exchange': 1, 'major': 1, 'holding': 0, 'periodic': 0}

PLANS = [
    Plan('A 전량 인덱싱',        ALL),
    Plan('B periodic 제외',      NO_PD),
    Plan('C T1+T2 부분',         T1_T2),
    Plan('D T1만',               T1),
    Plan('E T1만+개발 절제',     T1, dev_iterations=12, server_days=18),
    Plan('F E+호출수 절감',      T1, dev_iterations=12, server_days=18, calls_per_q=4),
]
report(PLANS)

---
## 5. 민감도 — 어느 미지수가 결론을 흔드는가

실측 우선순위를 정하기 위한 셀이다. 각 파라미터를 0.5배 / 2배로 흔들어
총비용이 얼마나 움직이는지 본다. **많이 움직이는 것부터 실측하라.**

In [ ]:
import copy

BASE_PLAN = Plan('base', T1, dev_iterations=12, server_days=18)
_globals = globals()

def with_param(name, factor, plan=BASE_PLAN):
    old = _globals[name]
    try:
        _globals[name] = old * factor
        c = scan_corpus() if name == 'CHARS_PER_TOKEN_KO' else CORPUS
        return total(plan, c)['total']
    finally:
        _globals[name] = old

base_total = total(BASE_PLAN)['total']
print(f'기준 총비용: {base_total:,.0f}원  (plan = {BASE_PLAN.name})\n')
print(f"{'parameter':<22}{'x0.5':>12}{'x1':>12}{'x2':>12}{'변동폭':>12}")
print('-' * 70)
rows = []
for name in ('PRICE_IN_PER_1K', 'PRICE_OUT_PER_1K', 'SERVER_KRW_PER_DAY', 'CHARS_PER_TOKEN_KO'):
    lo, hi = with_param(name, 0.5), with_param(name, 2.0)
    rows.append((name, lo, hi, abs(hi - lo)))
for name, lo, hi, span in sorted(rows, key=lambda r: -r[3]):
    print(f'{name:<22}{lo:>12,.0f}{base_total:>12,.0f}{hi:>12,.0f}{span:>12,.0f}')
print('-' * 70)
print('실측 우선순위 = 변동폭 큰 순')

---
## 6. 역산 — 남는 돈으로 인덱싱을 얼마나 할 수 있는가

개발·평가·서버를 먼저 확보하고, **남은 예산으로 감당 가능한 인덱싱 토큰 수**를 구한다.
이 숫자가 곧 "어느 티어까지 LLM 추출을 켤 수 있는가"의 답이다.

In [ ]:
def affordable_index_tokens(p, budget=None, reserve=0.15):
    """개발+평가+서버를 뺀 잔액으로 살 수 있는 인덱싱 입력토큰 수.
    reserve = 예비율(평가기간 재실행/실패 대비)."""
    budget = (budget or BUDGET_KRW) * (1 - reserve)
    c = cost(p)
    left = budget - (c['dev'] + c['eval'] + c['server'])
    if left <= 0:
        return 0.0, left
    per_tok = (PRICE_IN_PER_1K + p.out_ratio * PRICE_OUT_PER_1K) / 1000
    return left / per_tok, left

for p in (Plan('기본', T1), Plan('절제', T1, dev_iterations=12, server_days=18, calls_per_q=4)):
    toks, left = affordable_index_tokens(p)
    print(f'[{p.name}] 인덱싱 잔액 {left:,.0f}원  ->  감당 가능 입력토큰 {toks:,.0f}')
    if toks > 0:
        print('   유형별 커버리지:')
        for t, v in CORPUS.items():
            pct = min(100.0, 100 * toks / v['tokens'])
            mark = 'FULL' if pct >= 100 else f'{pct:5.1f}%'
            print(f'     {t:<10} {v["tokens"]:>12,.0f} tok   -> {mark}')
    print()

---
## 7. 결론 체크리스트

이 노트북을 돌린 뒤 확정해야 하는 것:

- [ ] **단가 4개 실측 완료** (`MEASURED` 전부 `True`) — 그전 금액은 전부 상대값이다
- [ ] **`periodic` LLM 추출은 끈다** — 전체 토큰의 95%. 목차 섹션 인덱스 + lazy 로딩으로 대체
- [ ] **개발셋 반복 횟수 상한** — §4에서 dev 비용이 무시할 수 없다는 게 드러난다.
      `dev_questions × dev_iterations`를 명시적으로 고정하라
- [ ] **질문당 호출 수 상한** — 파이프라인 설계에 하드코딩. 대화형 멀티에이전트를 피하는 이유
      ([../paper/multi_agent/multi_agent.md](../paper/multi_agent/multi_agent.md) §4)
- [ ] **서버 가동일 확정** — 8월 로컬 개발, 09-07~09-20만 필수 상시 가동
      ([../docs/clova_payment.md](../docs/clova_payment.md) §3)
- [ ] **예비율 15% 확보** — 평가기간 중 재실행·실패 대비
- [ ] **09-20 이후 서버 삭제** 캘린더 등록 — 미삭제 시 과금

### 조이는 순서 (품질 영향 적은 것부터)

1. 서버 가동일 — 규정·품질 영향 0
2. 개발셋 반복 횟수 — 실측 후 필요한 만큼만
3. 인덱싱 티어 — `periodic` → `holding` 순으로 끈다
4. 질문당 호출 수 — 여기부터는 품질에 영향이 온다
5. Self-Consistency `k` — Closed 질의 한정, 마지막에 조인다